<a href="https://colab.research.google.com/github/Mahendra2409/PyBlender/blob/main/Colab_Script/armadillo_ply_render.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>
<a href="https://drive.google.com/drive/folders/1sj-RqD5HRypGx-ZLqXpvyzY1CN84qJu-?usp=drive_link" target="_parent"><img src="https://img.shields.io/badge/PyBlender_Render_Farm-blue?logo=googledrive&logoColor=white" alt="PyBlender_Render_Farm"/></a>

# 🦔 Armadillo PLY Renderer

This notebook renders `.ply` format armadillo point cloud models using Blender's Cycles engine with a **ceramic material** pipeline.

### Pipeline Differences from `.xyz` Master Notebook
| Aspect | `.xyz` (master_render) | `.ply` (this notebook) |
|---|---|---|
| File format | `.xyz` (raw point coords) | `.ply` (mesh) |
| Loader | `bt.readNumpyPoints()` | `bt.readMesh()` |
| Material | Colormap-based point cloud | Ceramic (`bt.setMat_ceramic`) |
| Coloring | Distance-to-GT → colormap | Solid ceramic (derekBlue) |
| Shading | N/A | Smooth + Subdivision (level 2) |
| Post-process | N/A | Mix Shader Fac/Glossy disconnect |
| Ground truth | Required | Not used |

### Drive Data Path
```
📁 My Drive
 └── 📁 PyBlender_Render_Farm
      └── 📁 PointCloud
           └── 📁 plyFormat
                └── 📁 armadillo_PLY
                     ├── armadillo_denoised_bilateral_2.0.ply
                     ├── armadillo_noisy.ply
                     └── ... (other .ply files)
```

In [ ]:
#@title 1. Mount Drive
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# @title 2. Install Dependencies
!wget -nc -q https://download.blender.org/release/Blender4.0/blender-4.0.2-linux-x64.tar.xz
!tar -xf blender-4.0.2-linux-x64.tar.xz

!./blender-4.0.2-linux-x64/4.0/python/bin/python3.10 -m ensurepip --upgrade > /dev/null 2>&1

!./blender-4.0.2-linux-x64/4.0/python/bin/python3.10 -m pip install \
    scipy matplotlib numpy blendertoolbox \
    -q --no-input --disable-pip-version-check

In [ ]:
# @title 3. Download BlenderToolbox
!git clone https://github.com/HTDerekLiu/BlenderToolbox.git
!mv BlenderToolbox/blendertoolbox /content/
!./blender-4.0.2-linux-x64/4.0/python/bin/python3.10 -m pip install blendertoolbox

## Configuration Checklist

### Paths
- `DRIVE_BASE_PATH`: Root path on Google Drive
- `PC_TYPE`: Subfolder name inside `plyFormat/` (e.g. `armadillo_PLY`)

### Object Transforms
- `OBJ_LOCATION`, `OBJ_ROTATION`, `OBJ_SCALE`: Blender space transforms (must match the model)

### Material
- Ceramic material with `derekBlue` base color
- Smooth shading + subdivision level 2
- Mix Shader Fac & Glossy BSDF disconnected for cleaner output

In [ ]:
#@title 4. Config — armadillo_PLY
PLY_CONFIG = {

    # --- Output Controls ---
    "SAVE_BLEND_FILE": True,
    "FORCE_OVERWRITE": False,

    # --- Paths ---
    "DRIVE_BASE_PATH": "/content/drive/MyDrive/PyBlender_Render_Farm",
    "LOCAL_COLAB_BASE": "/content/local_data",
    "PC_TYPE": "armadillo_PLY",

    # --- Object Transforms ---
    "OBJ_LOCATION": (0.616392, -0.390241, -0.591646),
    "OBJ_ROTATION": (466.067, -2.68728, -306.609),
    "OBJ_SCALE": (0.006976, 0.006976, 0.006976),

    # --- Blender Render Settings ---
    "IMG_RES_X": 2000,
    "IMG_RES_Y": 2000,
    "NUM_SAMPLES": 100,
    "EXPOSURE": 1.5,
    "TILE_SIZE": 256,    # GPU tile size — smaller = less VRAM usage

    # --- Subdivision ---
    "SUBDIVISION_LEVEL": 2,

    # --- Camera Settings ---
    "CAM_LOCATION": (-1.9494, 1.5553, 0.71451),
    "LOOK_AT": (0, 0, 0),
    "FOCAL_LENGTH": 45,

    # --- Light Settings ---
    "LIGHT_ANGLE": (-17.5966, -47, -384),
    "LIGHT_STRENGTH": 2,
    "SHADOW_SOFTNESS": 0.3,
    "AMBIENT_COLOR": (0.1, 0.1, 0.1, 1),
    "SHADOW_THRESHOLD": 0.05,
}
print("✅ Config loaded: armadillo_PLY")


In [ ]:
#@title 5. Write config.py

def _ts(v):
    if isinstance(v, tuple): return '(' + ', '.join(str(x) for x in v) + ')'
    if isinstance(v, list): return '[' + ', '.join(str(x) for x in v) + ']'
    return repr(v)

with open('config.py', 'w') as f:
    f.write('# AUTO-GENERATED by armadillo_ply_render notebook\n\n')
    f.write('CONFIG = {\n')
    for ck, cv in PLY_CONFIG.items():
        f.write(f'    "{ck}": {_ts(cv)},\n')
    f.write('}\n')

print("✅ config.py written!")


In [ ]:
#@title 6. Core Rendering Logic (PLY Ceramic Pipeline)
%%writefile render_ply.py

import os, sys, shutil
import numpy as np

if '/content' not in sys.path:
    sys.path.append('/content')
from config import CONFIG

import bpy
import blendertoolbox as bt


def setup_env(CFG, src, local, out):
    """Copy .ply files from Drive to local SSD for faster I/O."""
    print("--- Setting up Environment ---")
    os.makedirs(out, exist_ok=True)
    if os.path.exists(local):
        shutil.rmtree(local)
    print("Copying PLY files from Drive to Local...")
    shutil.copytree(src, local)
    ply_files = [f for f in os.listdir(local) if f.endswith('.ply')]
    print(f"Copy complete! Found {len(ply_files)} .ply files.\n")


def render_single(CFG, meshPath, outputPath):
    """Render a single .ply file with ceramic material.

    Pipeline: readMesh → smooth shading → subdivision → ceramic material
              → disconnect Mix Shader Fac/Glossy → compositor denoise → render
    """
    ## initialize blender
    bt.blenderInit(CFG["IMG_RES_X"], CFG["IMG_RES_Y"], CFG["NUM_SAMPLES"], CFG["EXPOSURE"])

    ## GPU memory optimization — smaller tiles reduce VRAM usage
    bpy.context.scene.cycles.tile_size = CFG["TILE_SIZE"]

    ## read mesh
    mesh = bt.readMesh(meshPath, CFG["OBJ_LOCATION"], CFG["OBJ_ROTATION"], CFG["OBJ_SCALE"])

    ## smooth shading & subdivision
    bpy.ops.object.shade_smooth()
    bt.subdivision(mesh, level=CFG["SUBDIVISION_LEVEL"])

    ## set ceramic material
    meshC = bt.colorObj(bt.derekBlue, 0.5, 1.0, 1.0, 0.0, 0.0)
    subC = bt.colorObj(bt.derekBlue, 0.5, 2.0, 1.0, 0.0, 1.0)
    bt.setMat_ceramic(mesh, meshC, subC)

    ## remove some shader nodes for cleaner output
    # Disconnect Fac and Glossy BSDF from the final Mix Shader
    mat = bpy.context.object.active_material
    nodes = mat.node_tree.nodes
    links = mat.node_tree.links

    # Get the last Mix Shader node (connected to Material Output)
    mix_shader = next(
        (n for n in nodes if n.type == 'MIX_SHADER'
         and any(o.is_linked and o.links[0].to_node.type == 'OUTPUT_MATERIAL'
                 for o in n.outputs)),
        None
    )

    if mix_shader:
        # Disconnect Fac input
        if mix_shader.inputs['Fac'].is_linked:
            for link in mix_shader.inputs['Fac'].links:
                links.remove(link)

        # Disconnect second Shader input (Glossy BSDF)
        if mix_shader.inputs[2].is_linked:
            for link in mix_shader.inputs[2].links:
                links.remove(link)

        print("Disconnected Fac and Glossy BSDF from final Mix Shader.")
    else:
        print("Mix Shader node not found.")

    ## camera
    cam = bt.setCamera(CFG["CAM_LOCATION"], CFG["LOOK_AT"], CFG["FOCAL_LENGTH"])

    ## lighting
    sun = bt.setLight_sun(CFG["LIGHT_ANGLE"], CFG["LIGHT_STRENGTH"], CFG["SHADOW_SOFTNESS"])
    bt.setLight_ambient(color=CFG["AMBIENT_COLOR"])

    ## compositor (denoising setup)
    bpy.context.scene.use_nodes = True
    tree = bpy.context.scene.node_tree
    tree.nodes.clear()

    render_layers = tree.nodes.new('CompositorNodeRLayers')
    denoise_node = tree.nodes.new(type='CompositorNodeDenoise')
    composite = tree.nodes.new('CompositorNodeComposite')
    viewer = tree.nodes.new('CompositorNodeViewer')

    render_layers.location = (-300, 0)
    denoise_node.location = (0, 0)
    composite.location = (300, 0)
    viewer.location = (300, -200)

    tree.links.new(render_layers.outputs['Image'], denoise_node.inputs['Image'])
    tree.links.new(render_layers.outputs['Denoising Normal'], denoise_node.inputs['Normal'])
    tree.links.new(render_layers.outputs['Denoising Albedo'], denoise_node.inputs['Albedo'])
    tree.links.new(denoise_node.outputs['Image'], composite.inputs['Image'])
    tree.links.new(denoise_node.outputs['Image'], viewer.inputs['Image'])

    ## make gray shadow pure white (post-process)
    bt.shadowThreshold(alphaThreshold=CFG["SHADOW_THRESHOLD"], interpolationMode='CARDINAL')

    ## save .blend file
    if CFG["SAVE_BLEND_FILE"]:
        blend_path = outputPath.replace(".png", ".blend")
        bpy.ops.wm.save_mainfile(filepath=blend_path)
        print(f"Blend saved at {blend_path}")

    ## render image
    bt.renderImage(outputPath, cam)


def render_all(CFG, local, out):
    """Loop through all .ply files and render each one."""
    print("--- Starting Render Process ---")

    ply_files = sorted([f for f in os.listdir(local) if f.endswith('.ply')])
    total = len(ply_files)
    print(f"Found {total} .ply files to render.\n")

    for idx, filename in enumerate(ply_files, 1):
        meshPath = os.path.join(local, filename)
        outputPath = os.path.join(out, filename.replace(".ply", ".png"))

        if os.path.exists(outputPath) and not CFG["FORCE_OVERWRITE"]:
            print(f"[{idx}/{total}] Exists: {outputPath}. Skipping...")
            continue

        print(f"[{idx}/{total}] Rendering [{filename}]...")
        render_single(CFG, meshPath, outputPath)
        print(f"[{idx}/{total}] Render complete for {filename}\n")


if __name__ == "__main__":
    C = CONFIG

    print(f"\n{'='*60}")
    print(f"  RENDERING PLY: {C['PC_TYPE']}")
    print(f"{'='*60}\n")

    src = os.path.join(C["DRIVE_BASE_PATH"], "PointCloud", "plyFormat", C["PC_TYPE"])
    out = os.path.join(C["DRIVE_BASE_PATH"], "RenderImages", "plyFormat", C["PC_TYPE"])
    loc = os.path.join(C["LOCAL_COLAB_BASE"], C["PC_TYPE"])

    setup_env(C, src, loc, out)
    render_all(C, loc, out)

    print("\n🎉 All PLY rendering complete!")


In [ ]:
#@title 7. Start Rendering

!./blender-4.0.2-linux-x64/blender -b -P render_ply.py

In [ ]:
#@title 8. Extra: Preview Rendered Output
import os
from IPython.display import display, Image

DRIVE_BASE = "/content/drive/MyDrive/PyBlender_Render_Farm"
PC_TYPE = "armadillo_PLY"
output_dir = os.path.join(DRIVE_BASE, "RenderImages", "plyFormat", PC_TYPE)

if os.path.exists(output_dir):
    pngs = sorted([f for f in os.listdir(output_dir) if f.endswith('.png')])
    print(f"Found {len(pngs)} rendered images in {output_dir}")
    for png in pngs[:5]:  # Preview first 5
        print(f"\n--- {png} ---")
        display(Image(filename=os.path.join(output_dir, png), width=400))
else:
    print(f"Output directory not found: {output_dir}")
    print("Run the rendering cell first!")